# 03 · Production pipeline

**Story so far:** [02](./02_core_pipeline.ipynb) built the core pipeline. This
one is structurally identical — same envs, tasks, and flow — so you can diff
`02_core_pipeline.py` against `03_production_pipeline.py` and see exactly what
hardening adds. Each feature appears once; the pipeline's logic doesn't change.

**Covers Section 6 (Best Practices).**

**Flyte features**

1. **Reusable containers** — `flyte.ReusePolicy` keeps warm replicas
2. **Caching** — `cache="auto"` skips re-running on unchanged inputs
3. **Retries & timeouts** — `retries=` / `timeout=` per task
4. **Overrides** — `.override()` re-sizes and opts a single call out of reuse
5. **Traces** — `@flyte.trace` records a durable in-container call
6. **Reports** — `report=True` + `flyte.report` publish an HTML report

> **Demo:** run this twice — on the second run every `process_chunk` is a cache
> hit.

Run remotely with `flyte run 03_production_pipeline.py main`.

In [ ]:
from pathlib import Path

import flyte

# Points at this workshop's .flyte/config.yaml (demo cluster, org demo,
# project leon-demo). Swap in the Manas cluster's endpoint/org/project there.
flyte.init_from_config(Path(".flyte") / "config.yaml")

## 1. Reusable workers

By default each task call gets a fresh pod, and pod startup dominates the
runtime of short tasks. `flyte.ReusePolicy` keeps a pool of **warm replicas**
that accept successive calls, so a fan-out of many small tasks doesn't pay
startup cost each time. The knobs:

- **`replicas=(1, 2)`** — autoscale the warm pool between 1 and 2 replicas
- **`concurrency=10`** — task calls one replica handles at once (this is why
  async matters — concurrent calls share the replica)
- **`idle_ttl` / `scaledown_ttl`** — how long a replica lingers idle before
  it's reclaimed

Reuse requires `unionai-reuse` in the image — the only image change from
[02](./02_core_pipeline.ipynb). Replicas share memory across concurrent calls,
so size the pool with OOMs in mind. The driver env is unchanged.

In [ ]:
import asyncio
import os
from datetime import timedelta

import emoji  # installed locally AND declared in the image below
import flyte.report
from flyte.io import File
from pydantic import BaseModel

# Container images: reuse requires the unionai-reuse package.
image = flyte.Image.from_debian_base().with_pip_packages(
    "emoji", "unionai-reuse>=0.1.9"
)

# Reusable containers: ReusePolicy keeps replicas warm, so successive tasks
# skip pod startup.
worker_env = flyte.TaskEnvironment(
    name="prod_pipeline_worker",
    image=image,
    resources=flyte.Resources(cpu=2, memory="1Gi"),
    reusable=flyte.ReusePolicy(
        replicas=(1, 2),
        idle_ttl=60,
        concurrency=10,
        scaledown_ttl=60,
    ),
)

driver_env = flyte.TaskEnvironment(
    name="prod_pipeline_driver",
    image=image,
    resources=flyte.Resources(cpu=1, memory="500Mi"),
    depends_on=[worker_env],
    secrets=[flyte.Secret(key="ANTHROPIC_API_KEY", as_env_var="ANTHROPIC_API_KEY")],
)

## 2. Result and business logic (unchanged)

Identical to [02](./02_core_pipeline.ipynb). Hardening lives on the environments
and decorators — the model and helpers don't change, which is why the files
diff cleanly.

In [ ]:
class Summary(BaseModel):
    chunks: int
    lines: int
    words: int


def build_chunk_lines(chunk_id: int, lines: int) -> list[str]:
    sparkle = emoji.emojize(":sparkles:", language="alias")
    return [
        f"chunk {chunk_id}, line {i}: hello from flyte {sparkle}" for i in range(lines)
    ]


def count_words(text: str) -> int:
    return len(text.split())

## 3. Caching, retries & timeouts

All three are arguments to `@worker_env.task(...)`; the body is unchanged:

- **`cache="auto"`** — fingerprints inputs and code; a matching later call
  returns the stored output (why run 2 is all hits)
- **`retries=2`** — a failed attempt retries up to twice; each shows on the run page
- **`timeout=timedelta(minutes=5)`** — a hung task is killed after 5 min

`count_chunk_words` stays plain for contrast.

In [ ]:
# Caching, retries & timeouts: cached on inputs, retried on failure, and
# killed if it hangs past 5 minutes.
@worker_env.task(cache="auto", retries=2, timeout=timedelta(minutes=5))
async def process_chunk(chunk_id: int, lines: int = 10) -> list[str]:
    """Data I/O: plain typed values (list[str]) pass between tasks automatically."""
    result = [line.upper() for line in build_chunk_lines(chunk_id, lines)]
    print(f"chunk {chunk_id}: {len(result)} lines (only on cache miss)")
    return result


@worker_env.task
async def count_chunk_words(chunk: list[str]) -> int:
    if not chunk:
        raise ValueError("empty chunk")
    return count_words(" ".join(chunk))

## 4. Tolerant mapping (unchanged) and a cached merge

`tally_words` is the same tolerant `flyte.map` from
[02](./02_core_pipeline.ipynb). `summarize_and_archive` gains `cache="auto"` +
`retries=2` — deterministic in its inputs, so a re-run skips it.

In [ ]:
@driver_env.task
def tally_words(chunks: list[list[str]]) -> list[int]:
    """Mapping over inputs: flyte.map — one parallel action per input item."""
    tolerate_failures = False
    tolerate_failures = True  # <- comment out to fail the run and show the ValueError

    counts = []
    for result in flyte.map(
        count_chunk_words, [*chunks, []], return_exceptions=tolerate_failures
    ):
        if isinstance(result, Exception):
            print(f"skipping failed chunk: {result}")
        else:
            counts.append(result)
    return counts


# Caching: the merge step is cached and retried too.
@worker_env.task(cache="auto", retries=2)
async def summarize_and_archive(chunks: list[list[str]]) -> tuple[Summary, File]:
    """Files: merge the chunks, summarize, and archive the text as a File."""
    text = "\n".join(line for chunk in chunks for line in chunk)

    local_path = "/tmp/report.txt"
    Path(local_path).write_text(text)

    summary = Summary(
        chunks=len(chunks),
        lines=text.count("\n") + 1,
        words=count_words(text),
    )
    return summary, await File.from_local(local_path)

## 5. Traces

`@flyte.trace` (no parentheses) records a helper call as durable and observable
**without its own pod** — it runs in the caller's container but appears on the
run page with its I/O. Right for cheap glue calls (an LLM call, an annotation)
too small for a pod but worth capturing.

In [ ]:
# Traces: @flyte.trace (no parentheses) — a durable, observable call that runs
# inside the driver's container instead of its own pod.
@flyte.trace
async def annotate(summary: Summary) -> str:
    return f"{summary.chunks} chunks, {summary.lines} lines, {summary.words} words"

## 6. The driver: overrides + a report

Two additions over [02](./02_core_pipeline.ipynb)'s `main`:

- **`.override()`** — the last chunk is large. Rather than oversize the whole
  env, override just that call: `resources=...` bumps memory and `reusable="off"`
  gives it a dedicated pod (reusable replicas have fixed resources). The
  definition and other callers are untouched.
- **Reports** — `report=True` enables an HTML report; `flyte.report.replace.aio(...)`
  sets its content and `flush.aio()` publishes it as a tab on the run page. The
  `annotate` trace feeds it.

In [ ]:
@driver_env.task(report=True)
async def main(num_chunks: int = 4, heavy_lines: int = 1000) -> Summary:
    # Secrets: the secret arrives as a plain environment variable.
    print(f"Secret injected: {'ANTHROPIC_API_KEY' in os.environ}")

    # Fan-out: fan out over chunks — grouped on the run page, executed in parallel.
    with flyte.group("chunk-fanout"):
        coros = [process_chunk(i) for i in range(num_chunks)]
        # Overrides: the last chunk is known to be big — give that one call more
        # memory with .override() instead of oversizing the whole environment.
        # Reusable containers have fixed resources, so this call opts out of
        # reuse (reusable="off") to get its own right-sized pod.
        heavy = process_chunk.override(
            reusable="off", resources=flyte.Resources(cpu=2, memory="2Gi")
        )
        coros.append(heavy(num_chunks, lines=heavy_lines))
        chunks = await asyncio.gather(*coros)

    # Mapping: the same chunks again, this time fanned out with flyte.map.
    word_counts = tally_words(chunks)
    print(f"words per chunk: {word_counts}")

    # Tasks calling tasks: just another task call — its own action on the run page.
    summary, report_file = await summarize_and_archive(chunks)

    # Files: download the archived File before reading it like a local file.
    local_path = await report_file.download()
    first_line = Path(local_path).read_text().splitlines()[0]
    print(f"Archived {summary.lines} lines; first line: {first_line}")

    # Reports: publish an HTML report on the run page.
    note = await annotate(summary)
    await flyte.report.replace.aio(
        "<h2>Production Pipeline Report</h2>"
        f"<p>{note}</p>"
        f"<p>Archive: {report_file.path}</p>"
    )
    await flyte.report.flush.aio()

    return summary

## 7. Run it (twice)

First run fills the cache; run again and every `process_chunk` is a hit (no
"only on cache miss" print). On the run page: cached actions, the overridden
heavy chunk on its own non-reusable pod, the `annotate` trace, and the
**Report** tab.

In [ ]:
run = flyte.run(main, num_chunks=4)
print(f"Run URL: {run.url}")
run.wait()
run.outputs()

## Further reading

- Back to [02_core_pipeline](./02_core_pipeline.ipynb) to diff the un-hardened version
- Union docs: [caching](https://www.union.ai/docs/v2/union/user-guide/core-concepts/caching/) ·
  [reusable containers](https://www.union.ai/docs/v2/union/user-guide/task-configuration/reusable-containers/) ·
  [reports](https://www.union.ai/docs/v2/union/user-guide/core-concepts/reports/)